# 🏆 Notebook 6: The Complete AI-102 Pattern
---
### *Python for AI-102 — Azure AI Engineer Associate*

> **⏱️ Duration: ~1 Hour**

Congratulations! 🎉 You've learned all the Python building blocks. Now let's put everything together and see how a **real AI-102 lab script** is structured.

In this final notebook, we'll walk through complete code patterns that mirror exactly what you'll find in the Microsoft AI-102 GitHub lab repositories.

```
┌────────────────────────── AI-102 Lab Code Flow ──────────────────────┐
│                                                                       │
│   ┌─────────┐   ┌──────────┐   ┌──────────┐   ┌────────────────┐    │
│   │ Load    │──▶│ Create   │──▶│ Send     │──▶│ Process &     │    │
│   │ Config  │   │ Client   │   │ Request  │   │ Display Result│    │
│   └─────────┘   └──────────┘   └──────────┘   └────────────────┘    │
│                                                                       │
│   .env file      SDK Client     API Call        Print / Save          │
│   os.getenv()    credentials    text/image      Loop through          │
│                                                  results              │
└───────────────────────────────────────────────────────────────────────┘
```

## 📌 6.1 — Pattern 1: REST API Client (Text Analytics)

This pattern appears in the very first AI-102 lab. It uses the `requests` library to call Azure AI Services directly via REST.

Here's the actual structure you'll see:

In [ ]:
import os
import json

# =========================================================
# AI-102 Lab Pattern: REST API Client
# Service: Azure AI Language - Language Detection
# =========================================================

# ----- Configuration -----
# In real labs: from dotenv import load_dotenv; load_dotenv()
os.environ["AI_SERVICE_ENDPOINT"] = "https://eastus.api.cognitive.microsoft.com"
os.environ["AI_SERVICE_KEY"] = "demo-key-for-learning"

ai_endpoint = os.getenv("AI_SERVICE_ENDPOINT")
ai_key = os.getenv("AI_SERVICE_KEY")

def get_language(text):
    """
    Detect the language of the given text.
    Uses the Azure AI Language REST API.
    """
    # Build the request
    url = f"{ai_endpoint}/text/analytics/v3.1/languages"
    
    headers = {
        "Ocp-Apim-Subscription-Key": ai_key,
        "Content-Type": "application/json"
    }
    
    body = {
        "documents": [
            {"id": "1", "text": text}
        ]
    }
    
    # In real code:
    # response = requests.post(url, headers=headers, json=body)
    # return response.json()
    
    # Simulated response for learning
    language_map = {
        "hello": ("English", "en", 0.97),
        "bonjour": ("French", "fr", 0.95),
        "hola": ("Spanish", "es", 0.93),
        "guten tag": ("German", "de", 0.91),
    }
    
    first_word = text.lower().split()[0] if text else ""
    name, code, score = language_map.get(first_word, ("Unknown", "??", 0.50))
    
    return {
        "documents": [{
            "id": "1",
            "detectedLanguage": {
                "name": name,
                "iso6391Name": code,
                "confidenceScore": score
            }
        }]
    }


# ----- Main Program -----
def main():
    print("=" * 50)
    print("  Azure AI Language Detection (Simulated)")
    print("=" * 50)
    
    # Test texts
    test_texts = [
        "Hello, how are you today?",
        "Bonjour, comment allez-vous?",
        "Hola, buenos dias!",
        "Guten Tag, wie geht es Ihnen?"
    ]
    
    for text in test_texts:
        try:
            result = get_language(text)
            
            # Parse the response
            lang = result["documents"][0]["detectedLanguage"]
            
            print(f"\n  Text: \"{text}\"")
            print(f"  Language: {lang['name']} ({lang['iso6391Name']})")
            print(f"  Confidence: {lang['confidenceScore']:.0%}")
            
        except Exception as ex:
            print(f"\n  ❌ Error analyzing text: {ex}")

# Run the program
main()

## 📌 6.2 — Pattern 2: SDK Client (The Modern Way)

Most AI-102 labs now use the Azure **SDK** instead of raw REST calls. The SDK handles URL building, authentication, and response parsing for you.

Here's the pattern:

```
REST API way (manual):                SDK way (simpler):
──────────────────────                ─────────────────────
url = f"{endpoint}/..."               client = TextAnalyticsClient(
headers = {"Key": key}                    endpoint, credential
body = {"documents": [...]}           )
response = requests.post(...)         result = client.detect_language(
result = response.json()                  documents=[text]
                                      )
```

In [ ]:
import os

# =========================================================
# AI-102 Lab Pattern: SDK Client
# Service: Azure AI Language - Sentiment Analysis
# =========================================================

# ----- Configuration -----
os.environ["AI_SERVICE_ENDPOINT"] = "https://eastus.api.cognitive.microsoft.com"
os.environ["AI_SERVICE_KEY"] = "demo-key-for-learning"

ai_endpoint = os.getenv("AI_SERVICE_ENDPOINT")
ai_key = os.getenv("AI_SERVICE_KEY")

# In real AI-102 labs, you'd import the SDK:
# from azure.core.credentials import AzureKeyCredential
# from azure.ai.textanalytics import TextAnalyticsClient

# And create the client like this:
# credential = AzureKeyCredential(ai_key)
# client = TextAnalyticsClient(endpoint=ai_endpoint, credential=credential)


# ----- Simulated SDK Client -----
class SimulatedTextAnalyticsClient:
    """Simulates the Azure Text Analytics SDK client."""
    
    def __init__(self, endpoint, credential):
        self.endpoint = endpoint
        self.credential = credential
        print(f"  ✅ Client created for: {endpoint}")
    
    def analyze_sentiment(self, documents):
        """Simulate sentiment analysis."""
        results = []
        for doc in documents:
            text = doc.lower()
            if any(w in text for w in ["great", "love", "amazing", "excellent"]):
                results.append(SimulatedResult("positive", 0.92))
            elif any(w in text for w in ["bad", "terrible", "awful", "hate"]):
                results.append(SimulatedResult("negative", 0.88))
            else:
                results.append(SimulatedResult("neutral", 0.75))
        return results
    
    def extract_key_phrases(self, documents):
        """Simulate key phrase extraction."""
        results = []
        for doc in documents:
            words = doc.split()
            # Simple simulation: return 2-3 word phrases
            phrases = [" ".join(words[i:i+2]) for i in range(0, min(6, len(words)), 2)]
            results.append(SimulatedKeyPhraseResult(phrases))
        return results

class SimulatedResult:
    def __init__(self, sentiment, confidence):
        self.sentiment = sentiment
        self.confidence_scores = type('obj', (object,), {
            'positive': confidence if sentiment == "positive" else 0.05,
            'neutral': confidence if sentiment == "neutral" else 0.05,
            'negative': confidence if sentiment == "negative" else 0.05
        })()

class SimulatedKeyPhraseResult:
    def __init__(self, phrases):
        self.key_phrases = phrases


# ----- Main Program (exactly like AI-102 labs) -----
def main():
    # Create the client
    # Real: credential = AzureKeyCredential(ai_key)
    # Real: client = TextAnalyticsClient(endpoint=ai_endpoint, credential=credential)
    client = SimulatedTextAnalyticsClient(ai_endpoint, ai_key)
    
    # Documents to analyze
    documents = [
        "The Azure AI services are amazing and easy to use!",
        "I had a terrible experience with the deployment process.",
        "The documentation is available online for reference."
    ]
    
    # ----- Sentiment Analysis -----
    print("\n📊 Sentiment Analysis Results:")
    print("-" * 45)
    
    sentiments = client.analyze_sentiment(documents)
    
    for i, result in enumerate(sentiments):
        emoji = {"positive": "😊", "negative": "😞", "neutral": "😐"}
        print(f"  Doc {i+1}: {result.sentiment} {emoji.get(result.sentiment, '')}")
        print(f"    Positive:  {result.confidence_scores.positive:.2f}")
        print(f"    Neutral:   {result.confidence_scores.neutral:.2f}")
        print(f"    Negative:  {result.confidence_scores.negative:.2f}")
    
    # ----- Key Phrase Extraction -----
    print("\n🔑 Key Phrases:")
    print("-" * 45)
    
    phrases = client.extract_key_phrases(documents)
    
    for i, result in enumerate(phrases):
        print(f"  Doc {i+1}:")
        for phrase in result.key_phrases:
            print(f"    • {phrase}")

main()

## 📌 6.3 — Pattern 3: Computer Vision Lab

Vision labs follow a similar pattern but work with image files instead of text.

In [ ]:
import os
import json

# =========================================================
# AI-102 Lab Pattern: Computer Vision
# Service: Azure AI Vision - Image Analysis
# =========================================================

os.environ["AI_SERVICE_ENDPOINT"] = "https://eastus.api.cognitive.microsoft.com"
os.environ["AI_SERVICE_KEY"] = "demo-key-for-learning"

def analyze_image(image_path):
    """
    Analyze an image using Azure AI Vision.
    In real labs, this uses the Azure AI Vision SDK.
    """
    # Real SDK pattern:
    # from azure.ai.vision.imageanalysis import ImageAnalysisClient
    # from azure.ai.vision.imageanalysis.models import VisualFeatures
    # from azure.core.credentials import AzureKeyCredential
    #
    # client = ImageAnalysisClient(
    #     endpoint=os.getenv("AI_SERVICE_ENDPOINT"),
    #     credential=AzureKeyCredential(os.getenv("AI_SERVICE_KEY"))
    # )
    #
    # with open(image_path, "rb") as f:
    #     image_data = f.read()
    #
    # result = client.analyze(
    #     image_data=image_data,
    #     visual_features=[
    #         VisualFeatures.CAPTION,
    #         VisualFeatures.TAGS,
    #         VisualFeatures.OBJECTS,
    #         VisualFeatures.PEOPLE
    #     ]
    # )
    
    # Simulated result for learning
    return {
        "caption": {
            "text": "A person working on a laptop in a coffee shop",
            "confidence": 0.92
        },
        "tags": [
            {"name": "person", "confidence": 0.98},
            {"name": "laptop", "confidence": 0.95},
            {"name": "indoor", "confidence": 0.93},
            {"name": "table", "confidence": 0.87},
            {"name": "coffee", "confidence": 0.72}
        ],
        "objects": [
            {"name": "person", "confidence": 0.96, "boundingBox": {"x": 50, "y": 30, "w": 200, "h": 350}},
            {"name": "laptop", "confidence": 0.91, "boundingBox": {"x": 100, "y": 200, "w": 150, "h": 100}}
        ],
        "people": [
            {"confidence": 0.95, "boundingBox": {"x": 50, "y": 30, "w": 200, "h": 350}}
        ]
    }


# ----- Main Program -----
def main():
    image_path = "images/coffee_shop.jpg"
    
    print("🖼️  Azure AI Vision — Image Analysis")
    print("=" * 50)
    print(f"  Analyzing: {image_path}")
    
    try:
        result = analyze_image(image_path)
        
        # Display caption
        caption = result["caption"]
        print(f"\n📝 Caption:")
        print(f"  \"{caption['text']}\"")
        print(f"  Confidence: {caption['confidence']:.0%}")
        
        # Display tags
        print(f"\n🏷️  Tags:")
        for tag in result["tags"]:
            bar = "█" * int(tag["confidence"] * 20)
            print(f"  {tag['name']:15s} {bar} {tag['confidence']:.0%}")
        
        # Display detected objects
        print(f"\n📦 Objects:")
        for obj in result["objects"]:
            box = obj["boundingBox"]
            print(f"  {obj['name']:15s} ({obj['confidence']:.0%}) "
                  f"at [{box['x']}, {box['y']}]")
        
        # Display people count
        print(f"\n👥 People detected: {len(result['people'])}")
        
    except Exception as ex:
        print(f"\n❌ Error: {ex}")

main()

## 📌 6.4 — Pattern 4: Reading Multiple Files and Analyzing

This is a common lab pattern where you process all text files in a folder.

In [ ]:
import os
import json

# =========================================================
# AI-102 Lab Pattern: Batch Processing Files
# =========================================================

# Create sample data
os.makedirs("text-files", exist_ok=True)
files_data = {
    "review1.txt": "The Azure cognitive services are fantastic! Easy to set up and great documentation.",
    "review2.txt": "Had trouble with the API limits. Very frustrating experience overall.",
    "review3.txt": "The service works as expected. Nothing remarkable but it gets the job done.",
    "review4.txt": "Absolutely love the computer vision capabilities! Amazing accuracy."
}

for name, content in files_data.items():
    with open(os.path.join("text-files", name), "w") as f:
        f.write(content)

print("✅ Sample files created!")

# ----- Main Processing Logic -----
print("\n📊 Batch Text Analysis")
print("=" * 55)

folder = "text-files"
all_results = []

for filename in sorted(os.listdir(folder)):
    if filename.endswith(".txt"):
        filepath = os.path.join(folder, filename)
        
        # Read the file
        with open(filepath, "r") as f:
            text = f.read()
        
        # Simulate analysis (in real labs: client.analyze_sentiment([text]))
        text_lower = text.lower()
        if any(w in text_lower for w in ["fantastic", "love", "great", "amazing"]):
            sentiment, score = "positive", 0.91
        elif any(w in text_lower for w in ["trouble", "frustrating", "terrible"]):
            sentiment, score = "negative", 0.86
        else:
            sentiment, score = "neutral", 0.73
        
        result = {
            "file": filename,
            "sentiment": sentiment,
            "confidence": score,
            "text_preview": text[:50] + "..."
        }
        all_results.append(result)
        
        emoji = {"positive": "😊", "negative": "😞", "neutral": "😐"}
        print(f"\n  📄 {filename}")
        print(f"     {text[:60]}...")
        print(f"     → {sentiment} {emoji[sentiment]} ({score:.0%})")

# Summary
print("\n" + "=" * 55)
sentiments = [r["sentiment"] for r in all_results]
print(f"  📊 Summary: {len(all_results)} documents analyzed")
print(f"     Positive: {sentiments.count('positive')}")
print(f"     Negative: {sentiments.count('negative')}")
print(f"     Neutral:  {sentiments.count('neutral')}")

# Save results
with open("analysis_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\n  ✅ Results saved to analysis_results.json")

## 📌 6.5 — Quick Reference: Essential AI-102 Code Patterns

Here's a cheat sheet of the patterns you'll see across all AI-102 labs:

```python
# ═══════════════════════════════════════════════
# PATTERN: Setup & Configuration
# ═══════════════════════════════════════════════
from dotenv import load_dotenv
import os

load_dotenv()
endpoint = os.getenv("AI_SERVICE_ENDPOINT")
key = os.getenv("AI_SERVICE_KEY")

# ═══════════════════════════════════════════════
# PATTERN: Create SDK Client
# ═══════════════════════════════════════════════
from azure.core.credentials import AzureKeyCredential
credential = AzureKeyCredential(key)
client = SomeAzureClient(endpoint, credential)

# ═══════════════════════════════════════════════
# PATTERN: REST API Call
# ═══════════════════════════════════════════════
import requests
headers = {
    "Ocp-Apim-Subscription-Key": key,
    "Content-Type": "application/json"
}
response = requests.post(url, headers=headers, json=body)
result = response.json()

# ═══════════════════════════════════════════════
# PATTERN: Process Results
# ═══════════════════════════════════════════════
for item in result["documents"]:
    print(item["sentiment"])
    
# ═══════════════════════════════════════════════
# PATTERN: Error Handling
# ═══════════════════════════════════════════════
try:
    result = client.analyze(text)
except Exception as ex:
    print(f"Error: {ex}")
```

## 📌 6.6 — Libraries You'll Install in AI-102 Labs

Here's a summary of `pip install` commands you'll run:

| Command | What it installs |
|---------|-----------------|
| `pip install python-dotenv` | Load .env files |
| `pip install requests` | HTTP requests for REST APIs |
| `pip install azure-ai-textanalytics` | Text Analytics SDK |
| `pip install azure-ai-vision-imageanalysis` | Computer Vision SDK |
| `pip install azure-cognitiveservices-speech` | Speech SDK |
| `pip install azure-ai-formrecognizer` | Document Intelligence SDK |
| `pip install azure-search-documents` | Azure AI Search SDK |
| `pip install openai` | Azure OpenAI SDK |
| `pip install azure-identity` | Azure authentication |
| `pip install azure-core` | Core Azure utilities |
| `pip install pillow` | Image processing |
| `pip install matplotlib` | Charts and image display |

## 🎓 Course Summary — What You've Learned

Over these 6 notebooks (~6 hours), you've built up every Python skill needed for AI-102:

```
Notebook 1: 🐍 Foundations
  └── Variables, types, strings, f-strings, print, input

Notebook 2: 📦 Data Structures
  └── Lists, dictionaries, loops, nested data

Notebook 3: ⚙️ Functions & Modules
  └── def, if/else, import, os, json, try/except

Notebook 4: 📁 Files & Config
  └── Read/write files, .env, JSON files, binary files

Notebook 5: 🌐 REST APIs
  └── HTTP methods, requests, headers, responses, status codes

Notebook 6: 🏆 Complete AI-102 Patterns
  └── SDK clients, Vision labs, batch processing
```

### 🎯 You're Now Ready to:

1. **Read and understand** Python code in AI-102 lab guides
2. **Modify** code to change endpoints, keys, and parameters
3. **Debug** common issues (wrong key, missing file, bad JSON)
4. **Follow along** with any AI-102 lab exercise

### 📚 Next Steps:

1. Open the [AI-102 Microsoft Learn path](https://learn.microsoft.com/en-us/training/courses/ai-102t00)
2. Clone the lab repos from GitHub
3. Start with Lab 01 — you'll recognize all the patterns!

**Good luck on your AI-102 journey! 🚀🎉**